# 🌀 RayCash — Entraînement V2 (multi-dataset + anti-overfit)

**Améliorations vs V1** :
1. ➕ **Second dataset** Drinking Waste (via Kaggle API, optionnel) — +images d'aluminium/verre/plastique
2. 🧠 Backbone **EfficientNetV2-B0** (vs MobileNetV2) — meilleur ratio précision/taille
3. 🛡️ **Anti-overfit empilé** : dropout 0.4, L2 reg, label smoothing 0.1, BatchNorm, mixup, augmentation forte
4. ⚖️ **Class weighting** automatique : compense les classes sous-représentées (Aluminium, Inconnu)
5. 📉 **Cosine LR schedule** : meilleure convergence que constant LR
6. 📊 **Plot train vs val** : voir l'overfit à l'œil
7. ⏱️ Early-stopping généreux + restore_best_weights

Gain attendu : 80% → **88-92%** sur le test set, **avec écart train-val < 5 pts** (signe sain).

**Pré-requis Colab** : `Exécution → Modifier le type → T4 GPU` (gratuit).

## 0️⃣ Setup

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU :', gpus)
assert gpus, 'Active le GPU dans Exécution → Modifier le type d\'exécution'

In [ ]:
import os, json, shutil, random, pathlib, zipfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

WORK_DIR = pathlib.Path('/content/raycash')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

## 1️⃣ Dataset principal : TrashNet

In [ ]:
TRASHNET_URL = 'https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip'
TRASHNET_ZIP = WORK_DIR / 'trashnet.zip'
TRASHNET_DIR = WORK_DIR / 'dataset-resized'

if not TRASHNET_DIR.exists():
    print('Téléchargement TrashNet...')
    urllib.request.urlretrieve(TRASHNET_URL, TRASHNET_ZIP)
    with zipfile.ZipFile(TRASHNET_ZIP) as z:
        z.extractall(WORK_DIR)

for d in sorted(TRASHNET_DIR.iterdir()):
    if d.is_dir():
        print(f'  {d.name}: {len(list(d.glob("*.jpg")))} images')

## 2️⃣ Dataset secondaire : Drinking Waste (OPTIONNEL via Kaggle)

Active si tu as un compte Kaggle :
1. https://www.kaggle.com/settings → "Create New API Token" → `kaggle.json`
2. Dans Colab, panneau gauche **clé** → ajoute 2 secrets : `KAGGLE_USERNAME` et `KAGGLE_KEY` (depuis le json). Active "Notebook access".
3. Exécute la cellule.

Sinon **saute cette cellule** — le notebook continue avec TrashNet seul.

In [ ]:
WARP_DIR = WORK_DIR / 'warp'
WARP_ENABLED = False

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    !pip install -q kaggle
    if not WARP_DIR.exists():
        WARP_DIR.mkdir()
        !kaggle datasets download -d arkadiyhacks/drinking-waste-classification -p {WARP_DIR} --unzip
    WARP_ENABLED = True
    print('✅ Dataset secondaire chargé')
    for d in sorted(WARP_DIR.rglob('*')):
        if d.is_dir() and any(d.iterdir()):
            print(f'  {d.relative_to(WARP_DIR)}: {len(list(d.glob("*")))} fichiers')
except Exception as e:
    print(f'⚠️ Skip dataset secondaire ({type(e).__name__}: {e})')
    WARP_ENABLED = False

## 3️⃣ Mapping + fusion + split stratifié

In [ ]:
RAYCASH_CLASSES = ['Aluminium', 'Carton', 'Inconnu', 'Papier', 'Plastique', 'Verre']

TRASHNET_MAP = {
    'cardboard': 'Carton',
    'glass': 'Verre',
    'metal': 'Aluminium',
    'paper': 'Papier',
    'plastic': 'Plastique',
    'trash': 'Inconnu',
}
DRINKING_WASTE_MAP = {
    'Aluminium Cans':  'Aluminium',
    'Aluminum_Cans':   'Aluminium',
    'aluminum cans':   'Aluminium',
    'Glass':           'Verre',
    'Glass_Bottles':   'Verre',
    'Plastic':         'Plastique',
    'Plastic_Bottles': 'Plastique',
    'Paper Carton':    'Carton',
    'Paper_Cartons':   'Carton',
}

SPLIT_DIR = WORK_DIR / 'split'
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

TRAIN_RATIO, VAL_RATIO = 0.8, 0.1


def collect_images(root: pathlib.Path, mapping: dict[str, str]) -> dict[str, list[pathlib.Path]]:
    bucket: dict[str, list[pathlib.Path]] = {}
    if not root.exists():
        return bucket
    for src_dir in root.rglob('*'):
        if not src_dir.is_dir():
            continue
        target = None
        for key, val in mapping.items():
            if src_dir.name.lower() == key.lower():
                target = val
                break
        if target is None:
            continue
        images = list(src_dir.glob('*.jpg')) + list(src_dir.glob('*.jpeg')) + list(src_dir.glob('*.png'))
        if images:
            bucket.setdefault(target, []).extend(images)
    return bucket


buckets: dict[str, list[pathlib.Path]] = {c: [] for c in RAYCASH_CLASSES}
for cls, paths in collect_images(TRASHNET_DIR, TRASHNET_MAP).items():
    buckets[cls].extend(paths)
if WARP_ENABLED:
    for cls, paths in collect_images(WARP_DIR, DRINKING_WASTE_MAP).items():
        buckets[cls].extend(paths)

print('Comptage total par classe :')
for cls in RAYCASH_CLASSES:
    print(f'  {cls:12} {len(buckets[cls])}')

# Split stratifié : on découpe CHAQUE classe selon les ratios, pas le pool global
# → garantit que les classes faibles ont des samples partout (val + test).
for cls, files in buckets.items():
    random.Random(SEED).shuffle(files)
    n = len(files)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    splits = {
        'train': files[:n_train],
        'val':   files[n_train:n_train+n_val],
        'test':  files[n_train+n_val:],
    }
    for split_name, items in splits.items():
        dest = SPLIT_DIR / split_name / cls
        dest.mkdir(parents=True, exist_ok=True)
        for f in items:
            suffix = f.parent.parent.name + '_' + f.parent.name + '_' + f.name
            shutil.copy(f, dest / suffix)

for split in ['train', 'val', 'test']:
    total = 0
    print(f'\n[{split}]')
    for cls in RAYCASH_CLASSES:
        n = len(list((SPLIT_DIR / split / cls).glob('*')))
        total += n
        print(f'  {cls:12} {n}')
    print(f'  {"TOTAL":12} {total}')

## 4️⃣ Pipelines tf.data + augmentation FORTE + mixup

**Augmentation = défense #1 contre l'overfit** quand le dataset est petit. On augmente AGGRESSIVEMENT le train, on laisse val/test intacts.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = len(RAYCASH_CLASSES)
MIXUP_ALPHA = 0.2

def make_raw(split):
    return tf.keras.utils.image_dataset_from_directory(
        SPLIT_DIR / split,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_names=RAYCASH_CLASSES,
        shuffle=(split == 'train'),
        seed=SEED,
    )

train_raw = make_raw('train')
val_raw   = make_raw('val')
test_raw  = make_raw('test')
assert train_raw.class_names == RAYCASH_CLASSES

# Augmentation forte (mais réaliste — pas de RandomFlip vertical, les déchets sont posés à l'endroit)
data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

AUTOTUNE = tf.data.AUTOTUNE

# Mixup : combine 2 images aléatoires du batch → le modèle ne peut pas mémoriser un exemple unique
def mixup(batch_x, batch_y):
    batch_size = tf.shape(batch_x)[0]
    lam = tf.cast(tf.random.uniform([], 0.0, 1.0), tf.float32)
    lam = tf.maximum(lam, 1.0 - lam)
    idx = tf.random.shuffle(tf.range(batch_size))
    mixed_x = lam * batch_x + (1.0 - lam) * tf.gather(batch_x, idx)
    y_one = tf.one_hot(batch_y, NUM_CLASSES)
    y_mix = lam * y_one + (1.0 - lam) * tf.gather(y_one, idx)
    return mixed_x, y_mix

def prep_train(x, y):
    x = data_augment(x, training=True)
    return mixup(x, y)

def prep_eval(x, y):
    return x, tf.one_hot(y, NUM_CLASSES)

train_ds = train_raw.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_raw.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_raw.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

## 5️⃣ Class weights

In [ ]:
y_train_all = np.concatenate([y for _, y in train_raw], axis=0)
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train_all)
class_weight_dict = {i: float(w) for i, w in enumerate(weights)}
print('Class weights :')
for i, cls in enumerate(RAYCASH_CLASSES):
    print(f'  {cls:12} {class_weight_dict[i]:.3f}')

## 6️⃣ Modèle EfficientNetV2-B0 + défenses anti-overfit

**Défenses empilées** :
1. **Dropout 0.4** sur la tête (force la redondance des features)
2. **L2 regularization** 1e-4 sur le Dense final (pénalise les gros poids)
3. **BatchNorm** avant le Dropout (stabilise + régularise)
4. **Label smoothing 0.1** : le modèle apprend à ne pas être trop confiant
5. **Backbone gelé en phase 1** : pas de modif du pré-entraînement ImageNet (très anti-overfit pour petit dataset)
6. **Fine-tuning très limité en phase 2** : seulement les 20 dernières couches dégelées, LR très bas

In [ ]:
L2_REG = 1e-4
LABEL_SMOOTHING = 0.1
DROPOUT = 0.4

base = tf.keras.applications.EfficientNetV2B0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=True,
)
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(DROPOUT)(x)
outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation='softmax',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG),
)(x)
model = tf.keras.Model(inputs, outputs)

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=loss_fn,
    metrics=['accuracy'],
)
model.summary()

### Phase 1 — Head only (backbone gelé, LR 1e-3 avec cosine)

In [ ]:
# Early stopping : si val_accuracy ne progresse plus pendant 6 epochs, on stoppe et on restore les meilleurs poids
es = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=6,
    restore_best_weights=True,
    verbose=1,
)

EPOCHS_HEAD = 20
cosine_head = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=EPOCHS_HEAD * len(train_raw),
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(cosine_head),
    loss=loss_fn,
    metrics=['accuracy'],
)
history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
    callbacks=[es], class_weight=class_weight_dict,
)

### Phase 2 — Fine-tuning prudent (20 dernières couches uniquement, LR 5e-5)

On dégèle peu (20 couches) et avec un LR très bas pour ne pas casser les features pré-entraînées.

In [ ]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

EPOCHS_FT = 15
cosine_ft = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-5,
    decay_steps=EPOCHS_FT * len(train_raw),
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(cosine_ft),
    loss=loss_fn,
    metrics=['accuracy'],
)
history_ft = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
    callbacks=[es], class_weight=class_weight_dict,
)

## 7️⃣ Diagnostic d'overfit

**Ce qu'on veut voir** :
- train_acc et val_acc qui montent ensemble
- Écart final (train_acc − val_acc) **< 8 pts**
- val_loss qui descend ou stagne (pas qui remonte → overfit)

In [ ]:
def concat(a, b):
    return list(a) + list(b)

acc      = concat(history_head.history['accuracy'], history_ft.history['accuracy'])
val_acc  = concat(history_head.history['val_accuracy'], history_ft.history['val_accuracy'])
loss     = concat(history_head.history['loss'], history_ft.history['loss'])
val_loss = concat(history_head.history['val_loss'], history_ft.history['val_loss'])
phase_split = len(history_head.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(acc,     label='train', color='tab:blue')
axes[0].plot(val_acc, label='val',   color='tab:orange')
axes[0].axvline(phase_split - 0.5, color='gray', linestyle='--', label='start fine-tune')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(loss,     label='train', color='tab:blue')
axes[1].plot(val_loss, label='val',   color='tab:orange')
axes[1].axvline(phase_split - 0.5, color='gray', linestyle='--')
axes[1].set_title('Loss')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

final_gap = acc[-1] - val_acc[-1]
print(f'\n📊 Diagnostic overfit :')
print(f'  train_acc final : {acc[-1]*100:.2f}%')
print(f'  val_acc final   : {val_acc[-1]*100:.2f}%')
print(f'  écart           : {final_gap*100:+.2f} pts')
if final_gap > 0.08:
    print('  ⚠️ Écart > 8 pts : overfit. Augmente DROPOUT, L2_REG, ou réduis epochs.')
elif final_gap < 0.0:
    print('  🤔 Val > train : c\'est OK (mixup + regularisation rendent le train plus dur)')
else:
    print('  ✅ Sain : modèle généralise bien.')

## 8️⃣ Évaluation sur test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f'Test accuracy : {test_acc*100:.2f}%')

y_true, y_pred = [], []
for batch_x, _ in test_ds:
    probs = model.predict(batch_x, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1))
for _, batch_y in test_raw:
    y_true.extend(batch_y.numpy())

print(classification_report(y_true, y_pred, target_names=RAYCASH_CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=RAYCASH_CLASSES, yticklabels=RAYCASH_CLASSES)
plt.xlabel('Prédit'); plt.ylabel('Vérité')
plt.title('Matrice de confusion — RayCash V2')
plt.tight_layout()
plt.show()

## 9️⃣ Export TFLite int8

In [ ]:
def representative_dataset():
    for images, _ in train_raw.take(100 // BATCH_SIZE + 1):
        for img in images:
            yield [tf.cast(tf.expand_dims(img, 0), tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()
OUT_TFLITE = WORK_DIR / 'model.tflite'
OUT_TFLITE.write_bytes(tflite_model)
print(f'Modèle exporté : {OUT_TFLITE} ({OUT_TFLITE.stat().st_size / 1024:.0f} KB)')

OUT_LABELS = WORK_DIR / 'labels.txt'
OUT_LABELS.write_text('\n'.join(f'{i} {name}' for i, name in enumerate(RAYCASH_CLASSES)))
print('Labels :')
print(OUT_LABELS.read_text())

## 🔟 Sanity check TFLite + comparaison avec Keras

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(OUT_TFLITE))
interpreter.allocate_tensors()
in_det = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]
print('Input  :', in_det['shape'], in_det['dtype'])
print('Output :', out_det['shape'], out_det['dtype'])

correct_tflite, correct_keras, total = 0, 0, 0
for batch_x, batch_y in test_raw.unbatch().batch(1).take(300):
    y_gt = int(batch_y.numpy()[0])

    # TFLite
    x = tf.cast(batch_x, tf.uint8).numpy()
    interpreter.set_tensor(in_det['index'], x)
    interpreter.invoke()
    pred_tfl = int(np.argmax(interpreter.get_tensor(out_det['index'])[0]))

    # Keras (float)
    pred_k = int(np.argmax(model.predict(batch_x, verbose=0)[0]))

    correct_tflite += int(pred_tfl == y_gt)
    correct_keras  += int(pred_k   == y_gt)
    total += 1

print(f'\n300 samples test :')
print(f'  Keras (float32) : {correct_keras }/{total} = {correct_keras /total*100:.1f}%')
print(f'  TFLite (int8)   : {correct_tflite}/{total} = {correct_tflite/total*100:.1f}%')
print(f'  Perte due à la quantization : {(correct_keras-correct_tflite)/total*100:.1f} pts')

## 1️⃣1️⃣ Download

In [ ]:
from google.colab import files
files.download(str(OUT_TFLITE))
files.download(str(OUT_LABELS))